# 09 — Multi-Tenant Coexistence Demo

Demonstrates that multiple tenants coexist in the same Neptune graph with complete isolation.
Each tenant's nodes have labels like `__Type__tenant_id__` — no data leakage between tenants.


In [ ]:
# Setup
import json
from graphrag_toolkit.lexical_graph.storage import GraphStoreFactory
from document_graph.graph_build import node_to_cypher
from document_graph.query import DocumentGraphQueryEngine
from document_graph import Node

GRAPH_STORE = 'neptune-db://obs-app-dev-graph.cluster-czupj1ab2tc6.us-east-1.neptune.amazonaws.com:8182'

gs = GraphStoreFactory.for_graph_store(GRAPH_STORE).__enter__()
print('Neptune connected')


## Step 1: Write data for Tenant A


In [ ]:
TENANT_A = 'acme_corp'

tenant_a_data = [
    Node(id='u1', labels=['User'], properties={'name': 'Alice', 'role': 'admin', 'department': 'Engineering'}),
    Node(id='u2', labels=['User'], properties={'name': 'Bob', 'role': 'developer', 'department': 'Engineering'}),
    Node(id='p1', labels=['Project'], properties={'name': 'Apollo', 'status': 'active'}),
]

for node in tenant_a_data:
    cypher, params = node_to_cypher(node, tenant_id=TENANT_A)
    gs.execute_query(cypher, params)

print(f'Tenant A ({TENANT_A}): wrote {len(tenant_a_data)} nodes')


## Step 2: Write data for Tenant B


In [ ]:
TENANT_B = 'globex_inc'

tenant_b_data = [
    Node(id='u1', labels=['User'], properties={'name': 'Charlie', 'role': 'manager', 'department': 'Sales'}),
    Node(id='u2', labels=['User'], properties={'name': 'Diana', 'role': 'analyst', 'department': 'Finance'}),
    Node(id='p1', labels=['Project'], properties={'name': 'Zeus', 'status': 'planning'}),
]

for node in tenant_b_data:
    cypher, params = node_to_cypher(node, tenant_id=TENANT_B)
    gs.execute_query(cypher, params)

print(f'Tenant B ({TENANT_B}): wrote {len(tenant_b_data)} nodes')


## Step 3: Query — tenant isolation


In [ ]:
# Query Tenant A only
engine_a = DocumentGraphQueryEngine(gs, tenant_id=TENANT_A)
users_a = engine_a.get_nodes('User')
print(f'Tenant A Users: {len(users_a)}')
for u in users_a:
    props = u['n']['~properties']
    print(f'  {props.get("name")} ({props.get("role")})')

# Query Tenant B only
engine_b = DocumentGraphQueryEngine(gs, tenant_id=TENANT_B)
users_b = engine_b.get_nodes('User')
print(f'\nTenant B Users: {len(users_b)}')
for u in users_b:
    props = u['n']['~properties']
    print(f'  {props.get("name")} ({props.get("role")})')


## Step 4: Prove isolation — same ID, different tenants


In [ ]:
# Both tenants have node id='u1' but they're different nodes
a_u1 = engine_a.find_by_property('User', 'id', 'u1')
b_u1 = engine_b.find_by_property('User', 'id', 'u1')

a_name = a_u1[0]['n']['~properties']['name'] if a_u1 else 'NOT FOUND'
b_name = b_u1[0]['n']['~properties']['name'] if b_u1 else 'NOT FOUND'

print(f'Node u1 in Tenant A: {a_name}')
print(f'Node u1 in Tenant B: {b_name}')
print(f'\nSame ID, different data = tenant isolation working')


## Step 5: Cross-tenant view (admin only)


In [ ]:
# Admin can see all tenants by querying without tenant filter
all_users = gs.execute_query('MATCH (n) WHERE labels(n)[0] CONTAINS "User" RETURN labels(n) as lbl, properties(n) as props LIMIT 10')
print(f'All User nodes across tenants: {len(all_users)}')
for u in all_users:
    label = u['lbl'][0]
    name = u['props'].get('name', '?')
    # Extract tenant from label: __User__tenant__
    parts = label.split('__')
    tenant = parts[2] if len(parts) >= 3 else '?'
    print(f'  {name} (tenant: {tenant})')


## Summary

| Aspect | How it works |
|--------|-------------|
| **Isolation** | Labels encode tenant: `__User__acme_corp__` vs `__User__globex_inc__` |
| **Same IDs** | Node id='u1' exists in both tenants independently |
| **Querying** | `DocumentGraphQueryEngine(gs, tenant_id=X)` scopes all queries |
| **Admin view** | Query without tenant filter sees all data |
| **No leakage** | Tenant A cannot see Tenant B's data through normal queries |
